In [1]:
# MQA - Multi-Query Attention

import torch
import torch.nn as nn
import math


In [ ]:
class MQA(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        
        self.d_model = d_model
        self.num_heads = num_heads
        
        assert d_model % num_heads == 0, "d_model must divisible by num_heads"
        self.head_dim = d_model // num_heads
        
        self.query = nn.Linear(d_model, self.d_model)
        self.key = nn.Linear(d_model, self.head_dim) 
        self.val = nn.Linear(d_model, self.head_dim)
        
        self.linear = nn.Linear(d_model, d_model)
    
    def forward(self, x):
        batch_size, seq_len, d_model = x.size()

        query = self.query(x) 
        key = self.key(x).unsqueeze(1) # batch_size, 1, seq_len, head_dim
        val = self.val(x).unsqueeze(1) #batch_size, 1, seq_len, head_dim
        
        Q = query.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2)  # [batch_size, num_heads, seq_length, head_dim]
        print(f"Q.size: {Q.size()}")
        print(f"key.transpose(-2,-1).size(): {key.transpose(-2,-1).size()}")

        # attention
        scores = torch.matmul(Q, key.transpose(-2,-1))/ math.sqrt(d_model)
        print(scores.size())
        
        scores = torch.softmax(scores, dim=-1) # batch_size, seq_len, seq_len
        attn_output = torch.matmul(scores, val) #batch_size, num_heads, seq_len, head_dim
        
        attn_output = attn_output.transpose(1,2).contiguous().view(batch_size, seq_len, self.num_heads*self.head_dim)
        
        attn_output = self.linear(attn_output)
        
        return attn_output, scores

In [3]:
batch_size, seq_len, d_model = 16, 10, 768

mqa = MQA(d_model, 12)

x = torch.randn(batch_size, seq_len, d_model)

output, _ = mqa(x)

print(f"output is {output.size()}")

Q.size: torch.Size([16, 12, 10, 64])
key.transpose(-2,-1).size(): torch.Size([16, 1, 64, 10])
torch.Size([16, 12, 10, 10])
output is torch.Size([16, 10, 768])


In [4]:
# GQA - Grouped-Query Attention

class GQA(nn.Module):
    def __init__(self, d_model, head_dim, num_q_heads, num_kv_groups=None):
        super().__init__()
        self.d_model = d_model
        self.head_dim = head_dim
        self.num_kv_groups = num_kv_groups
        self.num_q_heads = num_q_heads
        
        assert num_q_heads % num_kv_groups==0, "num_q_heads must be divisible by num_kv_groups"

        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, num_kv_groups*head_dim)
        self.val = nn.Linear(d_model, num_kv_groups*head_dim)
        
        self.out_proj = nn.Linear(d_model, d_model)
        
    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        Q = self.query(x) # batch_szie, seq_len, d_model
        K = self.key(x)   # batch_size, seq_len, num_kv_groups*head_dim
        V = self.val(x)   # batch_szie, seq_len, num_kv_groups*head_dim

        # Q*KT
        # Q ：batch_size, num_q_heads, seq_len, head_dim
        # K: batch_size, num_kv_groups, seq_len, head_dim
        Q = Q.view(batch_size, seq_len, self.num_q_heads, self.head_dim).transpose(1,2)
        K = K.view(batch_size, seq_len, self.num_kv_groups, self.head_dim).transpose(1,2)
        K = torch.repeat_interleave(K, self.num_q_heads//self.num_kv_groups, 1) # batch_size, num_q_heads, seq_len, head_dim
        
        scores = torch.matmul(Q, K.transpose(-2,-1))/math.sqrt(self.head_dim)
        # batch_size, num_q_heads, seq_len, seq_len
        # V batch_szie, seq_len, num_kv_groups*head_dim
        
        V = V.view(batch_size, seq_len, self.num_kv_groups, self.head_dim).transpose(1,2)
        V = torch.repeat_interleave(V, self.num_q_heads//self.num_kv_groups, 1) #batch_size, num_q_heads, seq_len, head_dim
        scores = torch.softmax(scores, dim = -1)
        
        attn_out = torch.matmul(scores, V)
        attn_out = attn_out.transpose(1,2).contiguous().view(batch_size, self.seq_len, self.num_q_heads*self.head_dim)
        
        output = self.out_proj(attn_out)
        
        return output, scores

In [5]:
batch_size, num_kv_groups, seq_len, head_dim = 1,2,5,8
num_q_heads = 4
# batch_size, num_kv_groups, seq_len, head_dim ->
#batch_size, num_q_heads, seq_len, head_dim
x = torch.randn(batch_size, num_kv_groups, seq_len, head_dim)
x[0]

tensor([[[ 0.0177, -1.8175, -0.0276, -1.0684, -0.4260,  1.0987,  0.3518,
           0.1716],
         [-0.8593,  0.6436,  0.2640,  1.0305, -0.3117,  0.9487,  0.8488,
          -0.1725],
         [ 0.8921, -1.6099, -0.8887,  0.7766, -0.1761,  2.9487, -0.0958,
          -1.1721],
         [-1.3285, -0.4003, -0.3525,  0.2002,  0.1344,  0.4850, -1.1728,
          -0.5767],
         [-0.1031,  0.2141,  1.1117,  0.3099,  1.7202, -0.8457, -0.1135,
          -0.0302]],

        [[ 0.4109, -1.1289,  1.0138, -0.0920, -1.7533, -0.6742,  0.4559,
           0.9938],
         [-0.3840,  0.6367,  0.5781,  0.4757, -0.9515, -2.4222,  1.0240,
           1.9770],
         [ 0.2388, -0.6417, -0.5894,  0.7164,  1.9632, -0.0969, -0.0529,
           0.6436],
         [-1.7961,  0.0892,  0.1812, -0.7016,  1.2380, -1.7020, -0.5490,
          -1.6707],
         [ 0.1207, -1.3898,  0.5780, -0.9478, -0.8398,  0.2051, -0.2118,
           0.0585]]])

In [6]:
torch.repeat_interleave(x, 2, 1).size()

torch.Size([1, 4, 5, 8])

In [7]:
batch_size, seq_len, d_model = 16, 10, 768
gqa = GQA(d_model, 64 ,12, 4)

x = torch.randn(batch_size, seq_len, d_model)

output, _ = mqa(x)

print(f"output is {output.size()}")

Q.size: torch.Size([16, 12, 10, 64])
key.transpose(-2,-1).size(): torch.Size([16, 1, 64, 10])
torch.Size([16, 12, 10, 10])
output is torch.Size([16, 10, 768])


In [8]:
768//12

64